# Services: memory, resources, and pairs

Agent services are handles on `AgentContext`. They keep policy in the agent while delegating state or scheduling to the subsystem that owns it.


In [ ]:
from dataclasses import dataclass, field

from simyuj.components import PortKind
from simyuj.components.memories import MEMORY_EMIT, QuantumMemory
from simyuj.control import AgentContext, NodeAgent, SessionRuntime
from simyuj.engine import Component, Event, Timeline
from simyuj.entanglement import EntangledPairRecord, EntangledPairRegistry, PairState
from simyuj.network import Network, Node
from simyuj.network.routing import Route
from simyuj.network.topology import TopologyEdge
from simyuj.resources import MemoryRef, ResourceManager


Build one node-local agent with a memory device and shared resource/pair ledgers.


In [ ]:
@dataclass(slots=True)
class ServiceProbe(NodeAgent):
    contexts: list[AgentContext] = field(default_factory=list)

    def on_start(self, start, ctx: AgentContext) -> None:
        self.contexts.append(ctx)


In [ ]:
timeline = Timeline(master_seed=21)
network = Network("service_lab")

alice = Node("alice")
memory = QuantumMemory(memory_id="alice.mem", num_positions=2)
agent = ServiceProbe(agent_id="alice-controller", node_id="alice")

alice.add_device("mem", memory)
alice.add_device("plain", object())
alice.register_port("memory_notice", memory.notice_port)
alice.add_agent(agent)
network.add_node(alice)

resources = ResourceManager.from_network(network)
pairs = EntangledPairRegistry()

runtime = SessionRuntime(
    timeline=timeline,
    network=network,
    resource_manager=resources,
    pair_registry=pairs,
    session_id="service-session",
)
runtime.run()

ctx = agent.contexts[0]
print("Context agent:", ctx.agent_id)
print("Context node:", ctx.node_id)
print("Has devices service:", ctx.devices is not None)
print("Has memory service:", ctx.memory is not None)
print("Has resources service:", ctx.resources is not None)
print("Has pairs service:", ctx.pairs is not None)


Device lookup is node-local.


In [ ]:
print("Device 'mem' is:", type(ctx.devices.get("mem")).__name__)
print("Memory alias resolves to:", ctx.devices.memory("mem").memory_id)
print("Registered port alias:", ctx.devices.port("memory_notice").name)
print("Plain device type:", type(ctx.devices.get("plain")).__name__)


MemoryService schedules component requests. It does not run the memory handler immediately.


In [ ]:
before = timeline.events_scheduled
event = ctx.memory.emit("mem", 0, request_id="emit-demo")

print("Events before:", before)
print("Events after:", timeline.events_scheduled)
print("Event action:", event.action)
print("Target memory:", event.target_ref.memory_id)
print("Payload request id:", event.payload_ref.request_id)
print("Action is MEMORY_EMIT:", event.action == MEMORY_EMIT)


In [ ]:
print("Memory position state before running request:", memory.positions[0].status.value)
print("Pending event cancelled before execution:", event.cancelled)
timeline.cancel(event)
print("Pending event cancelled after cancellation:", event.cancelled)
print("Memory reports stored:", len(memory.reports))


ResourceService stamps reservations with the current agent id.


In [ ]:
available = ctx.resources.available_memories(timeline.current_time, "alice")
print("Available resource refs:", [ref.key for ref in available])

reservation = ctx.resources.reserve_memories(
    timeline.current_time,
    {"alice": 1},
    reservation_id="reservation:service-demo",
    created_at=timeline.current_time,
)

print("Reservation owner:", reservation.owner)
print("Reserved refs:", reservation.memory_ref_keys)
print("Reservation state:", reservation.state.value)


In [ ]:
committed = ctx.resources.commit(reservation.reservation_id)
ref = committed.memory_refs[0]
occupied = ctx.resources.mark_occupied(ref)

print("Committed state:", committed.state.value)
print("Occupied slot:", occupied.ref.key, occupied.state.value)
print("Reservation for ref:", ctx.resources.reservation_for_memory(ref).reservation_id)


PairService delegates to the entanglement registry.


In [ ]:
alice_ref = MemoryRef("alice", "mem", 0)
bob_ref = MemoryRef("bob", "mem", 0)

pair = ctx.pairs.register(
    EntangledPairRecord(
        "pair:service-demo",
        alice_ref,
        bob_ref,
        fidelity=0.92,
        created_at=timeline.current_time,
        expires_at=timeline.current_time + 5,
        generation_link_id="q_alice_bob",
    )
)

print("Registered pair:", pair.pair_id, pair.state.value)
print("Available Alice/Bob:", [p.pair_id for p in ctx.pairs.available_between("alice", "bob")])


In [ ]:
reserved_pair = ctx.pairs.reserve(pair.pair_id)
released_pair = ctx.pairs.release(pair.pair_id)

print("Reserved pair state:", reserved_pair.state.value)
print("Released pair state:", released_pair.state.value)
print("Current pair state:", ctx.pairs.get(pair.pair_id).state.value)


Pair expiry uses the runtime timeline's current time.


In [ ]:
class Noop(Component):
    def handle_event(self, event, timeline):
        return None

timeline.schedule(Event(time=timeline.current_time + 6, target_ref=Noop(), action="noop", payload_ref=None))
runtime.run_until_empty()

expired = ctx.pairs.expire_before_now()
print("Timeline now:", timeline.current_time)
print("Expired pairs:", [(p.pair_id, p.state.value) for p in expired])
print("Current pair state:", ctx.pairs.get(pair.pair_id).state.value)


Route helpers are available in context too.


In [ ]:
route_network = Network("route_probe")
for node_id in ("alice", "relay", "bob"):
    route_network.add_node(Node(node_id))
route_network.add_quantum_link("q_alice_relay", "alice", "relay")
route_network.add_quantum_link("q_relay_bob", "relay", "bob")

route = Route(
    "alice",
    "bob",
    (
        TopologyEdge("q_alice_relay", "alice", "relay", PortKind.QUANTUM),
        TopologyEdge("q_relay_bob", "relay", "bob", PortKind.QUANTUM),
    ),
)

print("Example route nodes:", route.node_ids)
print("Example route links:", route.link_ids)


The habit: services are thin handles. They either schedule timeline work or delegate to a ledger.
